# 11주차: DAG 보정집합 + 복합표본 회귀 (Table 2)
- 노출: 스트레스 인지(1=노출군, 0=비교군) / 결과: 하루 평균 수면시간(시간, 연속형) → **선형 회귀, β (95% CI)**
- 10주차 노트북의 불러오기·변수 만들기·`in_analysis` 코드를 **그대로** 가져온 뒤 3번 셀부터 실행합니다.
- '런타임 → 세션 다시 시작 및 모두 실행' 후 출력이 저장된 상태로 커밋합니다. 개인별 행은 출력하지 않습니다.

## 1. Drive 연결과 정의표의 변수만 읽기 (10주차와 같은 코드)

In [10]:
!pip -q install pyreadstat tabulate
import os
import numpy as np
import pandas as pd
import pyreadstat

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/hdac_data'
PATH = f'{DATA_DIR}/HN24_ALL.sav'
print(os.path.exists(PATH))


df, meta = pyreadstat.read_sav(PATH)
print(df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True
(6997, 813)


## 2. 노출·결과·공변량 만들기와 in_analysis (10주차와 같은 코드)

In [11]:

df['stress'] = np.select([df['BP1'].isin([1, 2]), df['BP1'].isin([3, 4])], [1, 0], default=np.nan)


WD = dict(bed_h='BP16_11', bed_m='BP16_12', wake_h='BP16_13', wake_m='BP16_14')   # 주중
WE = dict(bed_h='BP16_21', bed_m='BP16_22', wake_h='BP16_23', wake_m='BP16_24')   # 주말
SPECIAL = [88, 99]

def clean(col):
    return df[col].where(~df[col].isin(SPECIAL))

def sleep_hours(v):
    bed  = clean(v['bed_h'])  * 60 + clean(v['bed_m'])
    wake = clean(v['wake_h']) * 60 + clean(v['wake_m'])
    return ((wake - bed) % (24 * 60)) / 60

df['sleep_wd'] = sleep_hours(WD)
df['sleep_we'] = sleep_hours(WE)
df['sleep'] = (df['sleep_wd'] * 5 + df['sleep_we'] * 2) / 7


RAW = dict(
    sex='sex', age='age',
    marry='marri_1',      # 결혼 여부 (코드 1, 2)
    income='incm',        # 소득 4분위수
    edu='edu',            # 교육수준
    econ='EC1_1',         # 경제활동 상태
    phq9='mh_PHQ_S',      # 우울 PHQ-9 총점
    gad7='mh_GAD_S',      # 불안 GAD-7 점수
)
not_found = {k: v for k, v in RAW.items() if v not in df.columns}
if not_found:
    import re
    print("df 에 없는 변수:", not_found)
    labels = meta.column_names_to_labels
    for k in not_found:
        pat = {'marry':'결혼|혼인','income':'소득','edu':'교육','econ':'경제활동','phq9':'PHQ|우울','gad7':'GAD|불안'}.get(k, k)
        hits = [(c, labels.get(c)) for c in df.columns if re.search(pat, str(labels.get(c)) + c, re.I)]
        print(f"  [{k}] 후보:", hits[:10])
    raise KeyError("RAW 의 변수명을 위 후보로 고친 뒤 다시 실행하세요")
for new, old in RAW.items():
    if new not in ('sex', 'age'):
        df[new] = df[old]

df['in_analysis'] = (df['age'] >= 19) & df['stress'].notna() & df['sleep'].between(3, 16)
print("in_analysis n =", int(df['in_analysis'].sum()), "(Table 1 N = 5,800)")

in_analysis n = 5800 (Table 1 N = 5,800)


## 3. 설정 (10주차에서 만든 열 이름에 맞게 고치기)

In [15]:
CFG = dict(
    exposure='stress',
    outcome='sleep',
    weight='wt_itvex',
    strata='kstrata',
    psu='psu',
    adj_cat=['sex', 'marry', 'income', 'edu', 'econ'],
    adj_num=['age'],
    amb_num=['phq9', 'gad7'],
)

need = [CFG[k] for k in ['exposure', 'outcome', 'weight', 'strata', 'psu']] + CFG['adj_cat'] + CFG['adj_num'] + CFG['amb_num'] + ['in_analysis']
missing_cols = [c for c in need if c not in df.columns]
assert not missing_cols, f"missing columns: {missing_cols}"
assert set(df[CFG['exposure']].dropna().unique()) <= {0, 1}, "exposure must be 0/1"

## 4. 설계 기반 선형회귀 함수
가중치로 계수를 추정하고, 층·집락 선형화(Taylor) 분산으로 SE·95% CI를 구합니다. 행은 지우지 않고 분석 밖·결측 행은 가중치 0으로 둡니다(도메인 분석). 자유도 = 집락 수 − 층 수(전체 설계 기준).

In [16]:
from scipy import stats

def svy_ols(d, y, xcols, w, strata, psu):

    X = np.column_stack([np.ones(len(d))] + [d[c].to_numpy(float) for c in xcols])
    yy = d[y].to_numpy(float); ww = d[w].to_numpy(float)
    ok = ww > 0
    A = X[ok].T @ (X[ok] * ww[ok, None]); Ai = np.linalg.inv(A)
    beta = Ai @ (X[ok].T @ (ww[ok] * yy[ok]))
    e = np.where(ok, yy - X @ beta, 0.0)
    U = X * (ww * e)[:, None]
    key = d[strata].astype(str) + "|" + d[psu].astype(str)
    g = pd.DataFrame(U).groupby([d[strata].values, d[psu].values]).sum()   # 집락별 점수 합
    B = np.zeros_like(A); nh_tot = 0; npsu = 0
    for h, gh in g.groupby(level=0):
        n = len(gh); npsu += n; nh_tot += 1
        if n < 2: continue                      # 집락 1개 층은 분산 기여 없음(lonely PSU 제외)
        Z = gh.to_numpy() - gh.to_numpy().mean(0)
        B += n / (n - 1) * Z.T @ Z
    V = Ai @ B @ Ai
    df = npsu - nh_tot
    se = np.sqrt(np.diag(V)); t = beta / se; p = 2 * stats.t.sf(np.abs(t), df)
    q = stats.t.ppf(.975, df)
    names = ["const"] + list(xcols)
    return pd.DataFrame({"beta": beta, "se": se, "lo": beta - q*se, "hi": beta + q*se, "p": p}, index=names), int(ok.sum()), df

## 5. 보정변수가 모두 있는 사람(완전 사례) 표시와 인원

In [24]:
df['econ'] = df['econ'].where(df['econ'].isin([1, 2]))
d = df.copy()
inA = d['in_analysis'].astype(bool)
adj_all = CFG['adj_cat'] + CFG['adj_num']
cc2 = inA & d[adj_all].notna().all(axis=1)                       # 모형 1·2 공통 대상
cc3 = cc2 & d[CFG['amb_num']].notna().all(axis=1)                # 모형 3 대상
w0 = d[CFG['weight']].fillna(0)
d['w_cc2'] = np.where(cc2, w0, 0.0)
d['w_cc3'] = np.where(cc3, w0, 0.0)

n_t1 = int(inA.sum()); n2 = int(cc2.sum()); n3 = int(cc3.sum())
print(f"Table 1 분석 대상(in_analysis): {n_t1:,}명")
print(f"완전 사례(모형 1·2): {n2:,}명  (Table 1 대비 -{n_t1-n2:,}명)")
print(f"완전 사례(모형 3):   {n3:,}명  (Table 1 대비 -{n_t1-n3:,}명)")
print("보정변수별 결측 n (분석 대상 내):")
print(d.loc[inA, adj_all + CFG['amb_num']].isna().sum().to_string())
print("완전 사례 내 노출군/비교군 n:", d.loc[cc2, CFG['exposure']].value_counts().sort_index().to_dict())

Table 1 분석 대상(in_analysis): 5,800명
완전 사례(모형 1·2): 5,396명  (Table 1 대비 -404명)
완전 사례(모형 3):   5,386명  (Table 1 대비 -414명)
보정변수별 결측 n (분석 대상 내):
sex         0
marry       0
income     16
edu         3
econ      387
age         0
phq9      392
gad7        6
완전 사례 내 노출군/비교군 n: {0.0: 4079, 1.0: 1317}


## 6. 모형 1·2·3 적합 (설계 기반)

In [25]:
E, Y = CFG['exposure'], CFG['outcome']

def design(wcol, cols_cat, cols_num):
    inside = (d[wcol] > 0).to_numpy()
    X = pd.DataFrame({E: d[E].astype(float)})
    for c in cols_cat:
        cats = sorted(d.loc[inside, c].dropna().unique())
        for k in cats[1:]:
            X[f"{c}_{k}"] = (d[c] == k).astype(float)
    for c in cols_num:
        X[c] = d[c].astype(float)
    X = X.fillna(0.0)
    X[Y] = d[Y].fillna(0.0)
    X['_w'] = d[wcol].values
    X[CFG['strata']] = d[CFG['strata']].values
    X[CFG['psu']] = d[CFG['psu']].values
    return X

def fit(wcol, cols_cat, cols_num):
    X = design(wcol, cols_cat, cols_num)
    xs = [c for c in X.columns if c not in (Y, '_w', CFG['strata'], CFG['psu'])]
    res, n, dof = svy_ols(X, Y, xs, '_w', CFG['strata'], CFG['psu'])
    return res, n, dof

for c in CFG['adj_cat']:
    print(c, "ref =", sorted(d.loc[cc2, c].unique())[0], "n by category =", d.loc[cc2, c].value_counts().sort_index().to_dict())

m1, n1, df1 = fit('w_cc2', [], [])
m2, n2_, df2 = fit('w_cc2', CFG['adj_cat'], CFG['adj_num'])
m3, n3_, df3 = fit('w_cc3', CFG['adj_cat'], CFG['adj_num'] + CFG['amb_num'])
m2b, n2b, _ = fit('w_cc3', CFG['adj_cat'], CFG['adj_num'])
m1b, n1b, _ = fit('w_cc3', [], [])

assert n1 == n2_ == n2 and n3_ == n3, "N mismatch between models"
print("df (model 1,2 / model 3):", df1, "/", df3)

sex ref = 1.0 n by category = {1.0: 2308, 2.0: 3088}
marry ref = 1.0 n by category = {1.0: 4323, 2.0: 1073}
income ref = 1.0 n by category = {1.0: 1299, 2.0: 1363, 3.0: 1352, 4.0: 1382}
edu ref = 1.0 n by category = {1.0: 873, 2.0: 560, 3.0: 1772, 4.0: 2191}
econ ref = 1.0 n by category = {1.0: 3389, 2.0: 2007}
df (model 1,2 / model 3): 165 / 165


## 7. Table 2 형식으로 정리

In [26]:
def row(res, n):
    r = res.loc[E]
    return dict(beta=r.beta, lo=r.lo, hi=r.hi, p=r.p, n=n)

tbl = pd.DataFrame({
    'Model 1 (unadjusted)': row(m1, n2),
    'Model 2 (DAG adjusted)': row(m2, n2),
    'Model 3 (+PHQ9, GAD7)': row(m3, n3),
    'Ref: Model 1 in Model 3 sample': row(m1b, n3),
    'Ref: Model 2 in Model 3 sample': row(m2b, n3),
}).T

tbl['beta (95% CI)'] = tbl.apply(lambda r: f"{r.beta:.3f} ({r.lo:.3f}, {r.hi:.3f})", axis=1)
tbl['p'] = tbl['p'].map(lambda v: '<0.001' if v < 0.001 else f"{v:.3f}")
tbl['N analyzed'] = tbl['n'].astype(int).map('{:,}'.format)
tbl['Diff vs Table 1'] = (n_t1 - tbl['n'].astype(int)).map(lambda v: f"-{v:,}")

out = tbl[['beta (95% CI)', 'p', 'N analyzed', 'Diff vs Table 1']]
print(f"Table 1 analysis N = {n_t1:,}")
print(out.to_markdown())
print()
print("Change in beta, model 1 -> model 2:", f"{(m2.loc[E, 'beta'] / m1.loc[E, 'beta'] - 1) * 100:.1f}%")

Table 1 analysis N = 5,800
|                                | beta (95% CI)           | p      | N analyzed   |   Diff vs Table 1 |
|:-------------------------------|:------------------------|:-------|:-------------|------------------:|
| Model 1 (unadjusted)           | -0.146 (-0.228, -0.064) | <0.001 | 5,396        |              -404 |
| Model 2 (DAG adjusted)         | -0.187 (-0.269, -0.104) | <0.001 | 5,396        |              -404 |
| Model 3 (+PHQ9, GAD7)          | -0.129 (-0.226, -0.033) | 0.009  | 5,386        |              -414 |
| Ref: Model 1 in Model 3 sample | -0.145 (-0.227, -0.062) | <0.001 | 5,386        |              -414 |
| Ref: Model 2 in Model 3 sample | -0.185 (-0.268, -0.102) | <0.001 | 5,386        |              -414 |

Change in beta, model 1 -> model 2: 28.0%
